# 06 — Local LLM extraction
**Project:** Clinical Medication Extraction | **Phase 5a of the roadmap**

> ⚙️ **Runtime → GPU (T4 or better).** A 7B model in 4-bit needs ~6 GB VRAM.

## Why local models, not an API

This isn't a preference — it's the constraint that defines the project.

MIMIC's data use agreement prohibits sending note content to external LLM APIs. More broadly, no hospital will let PHI leave its network to a third-party endpoint. **Any clinical NLP system that depends on OpenAI or Anthropic APIs is undeployable in the environment you're targeting.**

So we develop against open-weight models from day one. Colab GPU here; Ollama on your Mac for the same prompts (verify parity once — same prompt, same model family, comparable output). When MIMIC access lands, the local path is already proven.

**This is also your strongest differentiator.** Most candidates' LLM projects are API wrappers. "I built this on local models because the data governance required it" is a sentence that lands in a healthcare interview, because it shows you designed for the constraint instead of discovering it later.

## What we're testing

The rules extractor and the NER model both failed in specific, documented ways:
- Negation scope: *"switched to ciprofloxacin without difficulty"* marked negated (wrong)
- Multi-drug segments: 7.5% of extractions with `attrs_ambiguous=True`, doses withheld
- Drugs outside the lexicon: found by NER but not normalizable

An LLM reads syntax. **The hypothesis is that it handles exactly these three cases better** — and we now have a harness that can prove or disprove it.

## Setup

In [1]:
%pip install -q transformers accelerate bitsandbytes torch

In [2]:
import pandas as pd
import numpy as np
import json, re, time
from collections import Counter

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    pass

BASE = '/content/drive/MyDrive/Clinical_notes/' if IN_COLAB else ''
WORK, SRC, GOLD = BASE + 'working/', BASE + 'src/', BASE + 'gold/'

import os, sys
sys.path.insert(0, SRC)
from sectionizer import split_sections
from evaluation import evaluate

work = pd.read_parquet(WORK + 'notes_subset.parquet')
ex_rules = pd.read_parquet(WORK + 'extractions_rules.parquet')

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL = 'Qwen/Qwen2.5-7B-Instruct'

bnb = BitsAndBytesConfig(load_in_4bit=True,
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_quant_type='nf4')

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map='auto')
print('Loaded', MODEL)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-7B-Instruct


**On 4-bit quantization:** weights are stored at 4 bits instead of 16, cutting memory ~4× with a small accuracy cost. It's what makes a 7B model fit on a free T4. Worth knowing the tradeoff exists — and worth noting in your model card, since quantization is a deployment decision with a measurable quality impact, not a free lunch.

---
# Part 1 — The prompt

## Design principles, each earning its place

1. **Schema stated explicitly, with the exact allowed status values.** Models invent plausible-sounding categories (`taking`, `current`, `prescribed`) unless constrained. Enumerating the vocabulary makes validation possible.
2. **"Return only JSON" and a one-shot example.** Format compliance improves dramatically with a concrete example — far more than with more instructions.
3. **An explicit rule against inference.** `Do not infer values that are not written` is the guardrail against the LLM's core failure mode: filling gaps with plausible clinical knowledge. A model that "knows" Lipitor is usually 20 mg will write 20 mg when the note says nothing.
4. **Status definitions inline**, matching your annotation guidelines exactly — so the LLM is scored against the same definitions your gold set uses.

In [4]:
SYSTEM_PROMPT = """You are a clinical information extraction system. You extract \
medication events from clinical notes and return them as JSON. You never infer \
information that is not explicitly written in the note."""

USER_TEMPLATE = """Extract every medication event from the clinical note below.

Return ONLY a JSON array. No preamble, no explanation, no markdown fences.

Each object must have exactly these keys:
  "drug"      - the medication name exactly as written in the note
  "dose"      - dose with units as written, or null if not stated
  "frequency" - frequency as written, or null if not stated
  "status"    - one of: active, discharge, allergy, historical, planned, negated, inpatient, mentioned

Status definitions:
  active     - patient is currently taking it
  discharge  - prescribed at discharge
  allergy    - listed as an allergy or intolerance
  historical - taken in the past, explicitly stopped
  planned    - recommended or to be started
  inpatient  - given during this admission only
  negated    - explicitly not taken ("denies", "not taking")
  mentioned  - referenced with no clear role

Rules:
- Do not infer values that are not written. Use null.
- Do not include IV fluids, blood products, or drug classes without a named agent.
- One object per distinct medication event.

Example output:
[{{"drug":"Lipitor","dose":"80 mg","frequency":"daily","status":"active"}},
 {{"drug":"Bactrim","dose":null,"frequency":null,"status":"allergy"}}]

CLINICAL NOTE:
{note}
"""

def build_prompt(note):
    return [{'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': USER_TEMPLATE.format(note=note)}]

print(USER_TEMPLATE.format(note='<note text here>')[:600])

Extract every medication event from the clinical note below.

Return ONLY a JSON array. No preamble, no explanation, no markdown fences.

Each object must have exactly these keys:
  "drug"      - the medication name exactly as written in the note
  "dose"      - dose with units as written, or null if not stated
  "frequency" - frequency as written, or null if not stated
  "status"    - one of: active, discharge, allergy, historical, planned, negated, inpatient, mentioned

Status definitions:
  active     - patient is currently taking it
  discharge  - prescribed at discharge
  allergy    - lis


In [5]:
@torch.inference_mode()
def generate(note, max_new_tokens=1024, temperature=0.0):
    text = tokenizer.apply_chat_template(build_prompt(note), tokenize=False,
                                         add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=temperature > 0, temperature=temperature or None,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

**Why `temperature=0.0`.** Temperature controls randomness in token selection. For extraction we want the same note to yield the same answer every time — reproducibility is a requirement, not a nicety, and any variance would contaminate your comparison with the rules baseline. Sampling is for creative generation; extraction is not creative.

---
# Part 2 — Parsing: assume the model will misbehave

**The single most important engineering lesson in this notebook.** LLMs return strings, not data structures. Even with "return only JSON," you will get markdown fences, preambles, dict wrappers, and occasionally truncated output.

**The mental model to adopt: an LLM in a pipeline is an unreliable but flexible component. Your job is wrapping it in validation until the *system* is reliable.** That reframe — from model quality to system reliability — is the most team-lead idea in the project.

In [6]:
def parse_llm_json(raw):
    """Parse model output into a list of dicts. Returns (records, failure_reason)."""
    if raw is None:
        return None, 'empty'
    text = re.sub(r'^```(?:json)?\s*', '', raw.strip())
    text = re.sub(r'\s*```$', '', text).strip()
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        start, end = text.find('['), text.rfind(']')       # salvage an embedded array
        if start == -1 or end == -1 or end < start:
            return None, 'no_array_found'
        try:
            obj = json.loads(text[start:end + 1])
        except json.JSONDecodeError:
            return None, 'malformed_json'
    if isinstance(obj, dict):                              # {"medications": [...]}
        for k in ('medications', 'drugs', 'results'):
            if isinstance(obj.get(k), list):
                obj = obj[k]
                break
        else:
            return None, 'dict_not_list'
    if not isinstance(obj, list):
        return None, 'not_a_list'
    return [r for r in obj if isinstance(r, dict)], None


REQUIRED = {'drug', 'dose', 'frequency', 'status'}
VALID_STATUS = {'active','discharge','allergy','historical','planned',
                'negated','inpatient','mentioned'}

def validate_record(rec):
    errs = []
    if REQUIRED - set(rec):
        errs.append(f'missing:{sorted(REQUIRED - set(rec))}')
    st = str(rec.get('status', '')).lower().strip()
    if st and st not in VALID_STATUS:
        errs.append(f'bad_status:{st}')
    if not str(rec.get('drug', '')).strip():
        errs.append('empty_drug')
    return errs


for name, raw in {
    'clean':        '[{"drug":"Lipitor","dose":"80 mg","frequency":"daily","status":"active"}]',
    'fenced':       '```json\n[{"drug":"aspirin","dose":null,"frequency":null,"status":"active"}]\n```',
    'preamble':     'Here are the medications:\n[{"drug":"warfarin","dose":null,"frequency":null,"status":"active"}]',
    'wrapped dict': '{"medications":[{"drug":"insulin","dose":"10 units","frequency":"at bedtime","status":"active"}]}',
    'broken':       '[{"drug":"aspirin", "dose": }]',
    'prose only':   'The patient takes aspirin daily.',
}.items():
    recs, err = parse_llm_json(raw)
    print(f'  {name:14} -> {"OK n=" + str(len(recs)) if recs is not None else "FAIL: " + err}')

  clean          -> OK n=1
  fenced         -> OK n=1
  preamble       -> OK n=1
  wrapped dict   -> OK n=1
  broken         -> FAIL: malformed_json
  prose only     -> FAIL: no_array_found


Four of six recovered; two correctly fail with a named reason. **Naming the failure matters** — `malformed_json` and `no_array_found` are different problems (one is truncation, usually fixed by raising `max_new_tokens`; the other is the model ignoring the format instruction, fixed by prompting). A generic `except: pass` would hide both.

In [7]:
def extract_with_retry(note, max_attempts=2):
    """Generate, parse, and retry once at higher temperature on failure."""
    for attempt in range(max_attempts):
        raw = generate(note, temperature=0.0 if attempt == 0 else 0.3)
        recs, err = parse_llm_json(raw)
        if recs is not None:
            clean = [r for r in recs if not validate_record(r)]
            return {'records': clean, 'raw': raw, 'attempts': attempt + 1,
                    'dropped_invalid': len(recs) - len(clean), 'error': None}
    return {'records': [], 'raw': raw, 'attempts': max_attempts,
            'dropped_invalid': 0, 'error': err}

**On the retry.** Attempt 1 is deterministic. If parsing fails, attempt 2 uses a small temperature — a different sample often lands on valid JSON where the greedy path got stuck. It's a cheap fix that materially reduces the failure rate.

Track `attempts` and `dropped_invalid` per note. **Those counters are your reliability metrics**, and they're what a monitoring dashboard would chart in production. Instrument early; retrofitting observability is painful.

---
# Part 3 — Faithfulness: measuring hallucination

The failure mode rules can't have and LLMs can: **outputting a drug that isn't in the note.** A model that knows clinical medicine will happily add the statin a patient "should" be on.

The check is simple and effective: **every extracted drug string must appear in the source note.** Normalize away case and punctuation, allow a prefix match for inflections and formulation suffixes, then flag anything left.

This is cheap, deterministic, and directly answers the question a clinician would ask first. It's also the metric that makes your rules baseline look good in a way that matters — **rules have a 0% hallucination rate by construction**, and that structural guarantee is a real argument in a safety-critical setting.

In [8]:
def _norm(s):
    return re.sub(r'[^a-z0-9]', '', str(s).lower())

def is_faithful(drug, note):
    if not drug:
        return False
    d, n = _norm(drug), _norm(note)
    if d and d in n:
        return True
    return len(d) >= 6 and d[:6] in n          # 'Lipitor XL' vs 'Lipitor'

def faithfulness_report(records, note):
    flags = [(r.get('drug'), is_faithful(r.get('drug'), note)) for r in records]
    bad = [d for d, ok in flags if not ok]
    return {'n': len(flags), 'hallucinated': len(bad),
            'rate': round(len(bad) / len(flags), 3) if flags else 0.0,
            'examples': bad[:5]}

demo_note = 'MEDICATIONS:, Lipitor 80 mg daily, aspirin 81 mg daily.'
demo_recs = [{'drug':'Lipitor'}, {'drug':'aspirin'}, {'drug':'metformin'}, {'drug':'Lipitor XL'}]
print(faithfulness_report(demo_recs, demo_note))

{'n': 4, 'hallucinated': 1, 'rate': 0.25, 'examples': ['metformin']}


`metformin` is correctly flagged; `Lipitor XL` correctly passes via prefix match. **Note the deliberate looseness** — a stricter check would flag `Lipitor XL` as hallucinated when the model actually did something reasonable. Every check has its own error rate, and choosing where to sit on that tradeoff is itself a decision to document.

---
# Part 4 — Run it

In [9]:
SAMPLE_N = 75                       # gold notes only — LLM inference is the slow step
gold_path = GOLD + 'gold_v1.csv'
if os.path.exists(gold_path):
    target_ids = pd.read_csv(gold_path)['note_id'].unique()
else:
    target_ids = work.sample(min(SAMPLE_N, len(work)), random_state=42).index

subset = work.loc[work.index.intersection(target_ids)]
print(f'Running LLM on {len(subset)} notes')

rows, meta = [], []
t0 = time.time()
for n, (note_id, note) in enumerate(subset['transcription'].items(), 1):
    out = extract_with_retry(note)
    faith = faithfulness_report(out['records'], note)
    meta.append({'note_id': note_id, 'attempts': out['attempts'],
                 'dropped_invalid': out['dropped_invalid'], 'error': out['error'],
                 'n_records': len(out['records']), 'hallucinated': faith['hallucinated']})
    for r in out['records']:
        rows.append({
            'note_id': note_id,
            'drug_text': r.get('drug'),
            'normalized': str(r.get('drug', '')).lower().strip(),
            'dose': r.get('dose'), 'frequency': r.get('frequency'),
            'status': str(r.get('status', 'mentioned')).lower().strip(),
            'faithful': is_faithful(r.get('drug'), note),
        })
    if n % 10 == 0:
        print(f'  {n}/{len(subset)}  ({(time.time()-t0)/n:.1f}s per note)')

ex_llm = pd.DataFrame(rows)
meta_df = pd.DataFrame(meta)
ex_llm.to_parquet(WORK + 'extractions_llm.parquet')
meta_df.to_csv(WORK + 'llm_run_meta.csv', index=False)

print()
print(f'{len(ex_llm)} extractions from {ex_llm["note_id"].nunique()} notes')
print(f'seconds per note      : {(time.time()-t0)/len(subset):.1f}')
print(f'needed a retry        : {(meta_df["attempts"] > 1).sum()} notes')
print(f'total parse failures  : {meta_df["error"].notna().sum()} notes')
print(f'invalid records dropped: {meta_df["dropped_invalid"].sum()}')
print(f'HALLUCINATION RATE    : {1 - ex_llm["faithful"].mean():.1%}')

Running LLM on 41 notes
  10/41  (8.7s per note)
  20/41  (10.6s per note)
  30/41  (13.0s per note)
  40/41  (12.8s per note)

271 extractions from 38 notes
seconds per note      : 13.2
needed a retry        : 0 notes
total parse failures  : 0 notes
invalid records dropped: 0
HALLUCINATION RATE    : 0.0%


### Reading these numbers

**Seconds per note** is your latency budget. Compare it to the rules extractor (milliseconds). If the LLM is 400× slower for a few points of F1, that ratio belongs in your ADR — and it's often the deciding argument for a hybrid.

**Retry and parse-failure counts** are reliability, not accuracy. A system that fails to parse 5% of the time has a 5% outage rate no matter how good the other 95% is.

**Hallucination rate** is the metric with no rules-baseline equivalent, because rules can't hallucinate. Frame it that way in your write-up: not "the LLM hallucinates 2%," but *"the LLM trades a structural 0% guarantee for a measured 2% rate, in exchange for X points of recall."* That's a tradeoff a hospital committee can actually evaluate.

---
# Part 5 — The three-way comparison

In [10]:
if os.path.exists(gold_path):
    gold = pd.read_csv(gold_path)
    gold_ids = set(gold['note_id'])

    preds = {'rules': ex_rules[ex_rules['note_id'].isin(gold_ids)].reset_index(drop=True),
             'llm':   ex_llm[ex_llm['note_id'].isin(gold_ids)].reset_index(drop=True)}
    ner_path = WORK + 'extractions_ner.parquet'
    if os.path.exists(ner_path):
        ner = pd.read_parquet(ner_path)
        preds['transformer'] = ner[ner['note_id'].isin(gold_ids)].reset_index(drop=True)

    table = {}
    for name, p in preds.items():
        r = evaluate(p, gold)['levels']
        table[name] = {f'{lv}_{m}': r[lv][m] for lv in r for m in ['precision','recall','f1']}
    comp = pd.DataFrame(table)[[c for c in ['rules','transformer','llm'] if c in table]]
    print(comp.to_string())
    comp.to_csv(WORK + 'comparison_three_way.csv')
else:
    print('gold_v1.csv not found — annotate first. Everything above runs without it.')

                       rules  transformer    llm
drug_precision         0.735        0.124  0.310
drug_recall            0.598        0.121  0.318
drug_f1                0.660        0.122  0.314
drug+status_precision  0.437        0.069  0.280
drug+status_recall     0.356        0.068  0.288
drug+status_f1         0.392        0.069  0.284


In [11]:
# Did the LLM fix the specific failures we predicted?
if os.path.exists(gold_path) and len(ex_llm):
    print('=== NEGATION: cases rules marked negated ===')
    neg_rules = ex_rules[(ex_rules['note_id'].isin(gold_ids)) & (ex_rules['status'] == 'negated')]
    for _, r in neg_rules.head(5).iterrows():
        m = ex_llm[(ex_llm['note_id'] == r['note_id'])
                   & (ex_llm['normalized'].str.contains(str(r['normalized'])[:6], case=False, na=False))]
        llm_status = m['status'].iloc[0] if len(m) else '(not found)'
        print(f"  {str(r['normalized'])[:22]:24} rules=negated  llm={llm_status}")
    print()
    print('=== DOSE RECOVERY: rules withheld attributes (multi-drug segments) ===')
    amb = ex_rules[(ex_rules['note_id'].isin(gold_ids)) & (ex_rules['attrs_ambiguous'])]
    got = 0
    for _, r in amb.iterrows():
        m = ex_llm[(ex_llm['note_id'] == r['note_id'])
                   & (ex_llm['normalized'].str.contains(str(r['normalized'])[:6], case=False, na=False))]
        if len(m) and pd.notna(m['dose'].iloc[0]):
            got += 1
    print(f'  rules withheld dose on {len(amb)} extractions; LLM supplied one on {got}')
    print('  -> verify a sample by hand: supplied is not the same as correct')

=== NEGATION: cases rules marked negated ===
  fluticasone              rules=negated  llm=(not found)
  warfarin                 rules=negated  llm=(not found)
  hydrocodone-acetaminop   rules=negated  llm=(not found)
  clopidogrel              rules=negated  llm=(not found)
  heparin                  rules=negated  llm=(not found)

=== DOSE RECOVERY: rules withheld attributes (multi-drug segments) ===
  rules withheld dose on 17 extractions; LLM supplied one on 0
  -> verify a sample by hand: supplied is not the same as correct


### The most important caution in this notebook

That last line matters. **The LLM supplying a dose where rules refused is not automatically an improvement.** Rules withheld because attribution was genuinely ambiguous. The LLM might resolve it correctly from syntax — or it might guess confidently.

Check a sample by hand against the note. If the LLM is right, that's a real capability and belongs in your ADR. If it's guessing, you've discovered that your baseline's "weakness" was actually appropriate caution, and the LLM's apparent advantage is confident nonsense.

**Only the manual check distinguishes those, and they lead to opposite architecture decisions.** This is the kind of analysis that separates someone reporting numbers from someone who can be trusted to own a clinical system.

In [12]:
llm_src = '''"""Local LLM extraction: parsing, validation, faithfulness."""
import json, re

REQUIRED = {"drug", "dose", "frequency", "status"}
VALID_STATUS = {"active","discharge","allergy","historical","planned",
                "negated","inpatient","mentioned"}


def parse_llm_json(raw):
    if raw is None:
        return None, "empty"
    text = re.sub(r"^```(?:json)?\\s*", "", raw.strip())
    text = re.sub(r"\\s*```$", "", text).strip()
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        start, end = text.find("["), text.rfind("]")
        if start == -1 or end == -1 or end < start:
            return None, "no_array_found"
        try:
            obj = json.loads(text[start:end + 1])
        except json.JSONDecodeError:
            return None, "malformed_json"
    if isinstance(obj, dict):
        for k in ("medications", "drugs", "results"):
            if isinstance(obj.get(k), list):
                obj = obj[k]
                break
        else:
            return None, "dict_not_list"
    if not isinstance(obj, list):
        return None, "not_a_list"
    return [r for r in obj if isinstance(r, dict)], None


def validate_record(rec):
    errs = []
    if REQUIRED - set(rec):
        errs.append("missing:" + str(sorted(REQUIRED - set(rec))))
    st = str(rec.get("status", "")).lower().strip()
    if st and st not in VALID_STATUS:
        errs.append("bad_status:" + st)
    if not str(rec.get("drug", "")).strip():
        errs.append("empty_drug")
    return errs


def _norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())


def is_faithful(drug, note):
    if not drug:
        return False
    d, n = _norm(drug), _norm(note)
    if d and d in n:
        return True
    return len(d) >= 6 and d[:6] in n


def faithfulness_report(records, note):
    flags = [(r.get("drug"), is_faithful(r.get("drug"), note)) for r in records]
    bad = [d for d, ok in flags if not ok]
    return {"n": len(flags), "hallucinated": len(bad),
            "rate": round(len(bad) / len(flags), 3) if flags else 0.0,
            "examples": bad[:5]}
'''

with open(SRC + 'llm_extractor.py', 'w') as f:
    f.write(llm_src)

import importlib.util
spec = importlib.util.spec_from_file_location('llm_extractor', SRC + 'llm_extractor.py')
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
assert mod.parse_llm_json('```json\n[{"drug":"x"}]\n```')[0] == [{'drug': 'x'}]
assert not mod.is_faithful('metformin', 'patient takes aspirin')
print('Wrote src/llm_extractor.py and verified it.')

Wrote src/llm_extractor.py and verified it.


---
## What you built

A local LLM extractor with a constrained prompt, defensive parsing with named failure modes, a retry loop, record validation, a faithfulness check, per-note reliability instrumentation, and the three-way comparison.

**The four transferable ideas:**
1. **Local models are the design constraint, not a compromise.** APIs are undeployable with PHI; building local from day one is what makes this portfolio-to-hospital transferable.
2. **Wrap unreliable components until the system is reliable.** Parse, salvage, validate, retry, count. The model is a component; reliability is the system's property.
3. **Measure hallucination explicitly.** Rules have a structural 0% guarantee. Quantify what you're giving up for what you're gaining.
4. **"Supplied a value" ≠ "supplied the right value."** Where the baseline abstained deliberately, only manual review tells you whether the LLM resolved or guessed.

**For `decisions.md`:**
- Qwen2.5-7B-Instruct 4-bit local; external APIs excluded by DUA/PHI constraints — architecture decision, not preference
- temperature=0.0 for reproducibility; retry at 0.3 only on parse failure
- Parse failures named (`malformed_json` vs `no_array_found`) — different causes, different fixes
- Faithfulness = extracted drug string must appear in note; prefix match tolerated for formulations
- Latency and retry rate tracked per note as reliability metrics
- Dose recovery on previously-ambiguous cases requires manual verification before being claimed as a gain

**Next: `07_rag_normalization.ipynb`** — retrieval over RxNorm so `Toprol XL` and `Ramelteon` resolve to canonical concepts, evaluated separately from end-to-end accuracy.